# CNN baseline — ASL-HG, participant-disjoint

Baseline tái lập: archive processed của ASL-HG, hash metadata đã publish, khử exact duplicate và split theo người tham gia (8/1/1). Không dùng split train/test dựng sẵn của tác giả vì protocol này khóa validation và test theo participant.


In [ ]:
%pip -q install 'huggingface-hub>=0.25' pandas pyarrow scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, random, re, shutil, zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from huggingface_hub import snapshot_download
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

EXPERIMENT_ID = 'cnn-001-participant-disjoint'
DATASET_REPO = 'hnam25/asl-hand-gesture-images'
DATASET_REVISION = '8f36ac00ece6dfce94410a980a839d93a912d366'
AUDIT_DIRECTORY = 'metadata/colab-audit-2026-08-10'
PROCESSED_ARCHIVE = 'ASL_HG_36000/ASL_Processed_Images.zip'
SEED, IMAGE_SIZE, BATCH_SIZE, EPOCHS, LEARNING_RATE, DROPOUT = 42, 128, 64, 30, 1e-3, .3
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
ROOT = Path('/content/asl-cnn-baseline')
HF_ROOT, PROCESSED, OUTPUTS = ROOT/'hf', ROOT/'processed', ROOT/'outputs'
for directory in (HF_ROOT, PROCESSED, OUTPUTS/'models', OUTPUTS/'metrics', OUTPUTS/'figures', OUTPUTS/'logs', OUTPUTS/'metadata'):
    directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print({'tensorflow': tf.__version__, 'gpus': [d.name for d in tf.config.list_physical_devices('GPU')], 'experiment_id': EXPERIMENT_ID})


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION, local_dir=HF_ROOT, allow_patterns=[PROCESSED_ARCHIVE, f'{AUDIT_DIRECTORY}/**'])
archive = HF_ROOT / PROCESSED_ARCHIVE
audit_root = HF_ROOT / AUDIT_DIRECTORY
audit_manifest = json.loads((audit_root/'cache_manifest.json').read_text())
if not archive.is_file(): raise RuntimeError(f'Missing {PROCESSED_ARCHIVE}')
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (PROCESSED/member.filename).resolve()
        if PROCESSED.resolve() not in target.parents and target != PROCESSED.resolve(): raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    z.extractall(PROCESSED)
processed_archive_sha256 = sha256_file(archive)
audit = pd.read_csv(audit_root/'audit.csv')
print({'processed_archive_sha256': processed_archive_sha256, 'audit_totals': audit_manifest['totals']})


In [ ]:
# Match every processed image to audited raw-image hash, then retain one representative per exact hash.
image_paths = sorted(path for path in PROCESSED.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
records = []
subject_pattern = re.compile(r'^P(\d+)_')
for path in image_paths:
    parts = path.parts
    label = next((part for part in reversed(parts[:-1]) if part in CLASSES), None)
    if label is None: raise RuntimeError(f'Cannot infer label from {path}')
    match = subject_pattern.match(path.name)
    if not match: raise RuntimeError(f'Cannot infer participant from {path.name}')
    records.append({'label': label, 'processed_path': str(path), 'relative_path': f'{label}/{path.name}', 'participant': f'P{match.group(1)}'})
processed = pd.DataFrame(records)
usable = processed.merge(audit[audit.status == 'ok'][['relative_path', 'label', 'sha256']], on=['relative_path', 'label'], how='left', validate='one_to_one')
if len(usable) != 36000 or usable.sha256.isna().any(): raise RuntimeError('Processed archive does not match the pinned audit metadata.')
if (usable.groupby('sha256').label.nunique() > 1).any(): raise RuntimeError('Conflicting labels for an exact raw-image hash.')
usable = usable.sort_values(['sha256', 'relative_path'], kind='stable').reset_index(drop=True)
usable['duplicate_group_size'] = usable.groupby('sha256').sha256.transform('size')
usable['canonical_relative_path'] = usable.groupby('sha256').relative_path.transform('first')
usable['is_canonical'] = usable.relative_path.eq(usable.canonical_relative_path)
usable.to_csv(OUTPUTS/'metadata'/'deduplication_manifest.csv', index=False)
before_dedup = len(usable); usable = usable[usable.is_canonical].copy()
participants = sorted(usable.participant.unique(), key=lambda value: int(value[1:]))
if len(participants) != 10 or set(usable.participant) != set(participants): raise RuntimeError(f'Expected P1-P10, got {participants}')
rng = np.random.default_rng(SEED); ordered = list(rng.permutation(participants))
train_participants, validation_participant, test_participant = sorted(ordered[:8]), ordered[8], ordered[9]
train = usable[usable.participant.isin(train_participants)].copy()
validation = usable[usable.participant.eq(validation_participant)].copy()
test = usable[usable.participant.eq(test_participant)].copy()
for name, frame in {'train': train, 'validation': validation, 'test': test}.items(): frame[['processed_path', 'label', 'participant', 'sha256']].to_csv(OUTPUTS/'metadata'/f'{name}.csv', index=False)
if set(train.sha256) & set(validation.sha256) or set(train.sha256) & set(test.sha256) or set(validation.sha256) & set(test.sha256): raise RuntimeError('Hash leakage detected.')
if set(train.participant) & set(validation.participant) or set(train.participant) & set(test.participant) or set(validation.participant) & set(test.participant): raise RuntimeError('Participant leakage detected.')
split_manifest = {'experiment_id': EXPERIMENT_ID, 'policy': 'participant-disjoint 8/1/1; one canonical representative per exact raw-image SHA-256', 'seed': SEED, 'participants': {'train': train_participants, 'validation': validation_participant, 'test': test_participant}, 'counts': {'before_deduplication': before_dedup, 'after_deduplication': len(usable), 'removed_exact_duplicates': before_dedup-len(usable), 'train': len(train), 'validation': len(validation), 'test': len(test)}}
(OUTPUTS/'metadata'/'split_manifest.json').write_text(json.dumps(split_manifest, indent=2), encoding='utf-8')
print(json.dumps(split_manifest, indent=2))


In [ ]:
label_index = {label: index for index, label in enumerate(CLASSES)}
def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.processed_path.values, frame.label.map(label_index).values))
    if training: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def load(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False); image.set_shape([None, None, 3])
        image = tf.image.resize(tf.cast(image, tf.float32), (IMAGE_SIZE, IMAGE_SIZE))
        return image, tf.one_hot(label, len(CLASSES))
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, validation_ds, test_ds = make_dataset(train, True), make_dataset(validation), make_dataset(test)
def build_cnn():
    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
    x = tf.keras.layers.Rescaling(1/255, name='rescale')(inputs)
    for filters in (32, 64, 128):
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x); x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x); x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPooling2D()(x); x = tf.keras.layers.Dropout(DROPOUT / 2)(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
    x = tf.keras.layers.Dense(256, activation='relu', name='features')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout')(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax', name='classification')(x)
    return tf.keras.Model(inputs, outputs, name='cnn_asl')

model = build_cnn(); model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='categorical_crossentropy', metrics=['accuracy'])
with (OUTPUTS/'models'/'model_summary.txt').open('w') as handle: model.summary(print_fn=lambda line: handle.write(line+'\n'))
checkpoint = OUTPUTS/'models'/'cnn_001_participant_disjoint.keras'
callbacks = [tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_accuracy', save_best_only=True), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=.2), tf.keras.callbacks.CSVLogger(OUTPUTS/'logs'/'training_history.csv')]
history = model.fit(train_ds, validation_data=validation_ds, epochs=EPOCHS, verbose=2, callbacks=callbacks)
pd.DataFrame(history.history).to_csv(OUTPUTS/'logs'/'training_history.csv', index=False)


In [ ]:
best = tf.keras.models.load_model(checkpoint)
probability = best.predict(test_ds, verbose=1); predicted = probability.argmax(1); truth = test.label.map(label_index).to_numpy()
report = classification_report(truth, predicted, labels=range(36), target_names=CLASSES, output_dict=True, zero_division=0)
matrix = confusion_matrix(truth, predicted, labels=range(36))
pd.DataFrame(report).T.to_csv(OUTPUTS/'metrics'/'classification_report.csv')
pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(OUTPUTS/'metrics'/'confusion_matrix.csv')
o, zero = label_index['O'], label_index['0']
summary = {'experiment_id': EXPERIMENT_ID, 'test_accuracy': float(accuracy_score(truth, predicted)), 'macro_precision': report['macro avg']['precision'], 'macro_recall': report['macro avg']['recall'], 'macro_f1': report['macro avg']['f1-score'], 'O_recall': report['O']['recall'], '0_recall': report['0']['recall'], 'O_to_0': int(matrix[o, zero]), '0_to_O': int(matrix[zero, o]), 'best_validation_accuracy': float(max(history.history['val_accuracy'])), 'epochs_ran': len(history.history['loss'])}
(OUTPUTS/'metrics'/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
plt.figure(figsize=(16, 13)); sns.heatmap(matrix, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig(OUTPUTS/'figures'/'confusion_matrix.png', dpi=180); plt.close()
config = {'experiment_id': EXPERIMENT_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset_repo': DATASET_REPO, 'dataset_revision': DATASET_REVISION, 'processed_archive': PROCESSED_ARCHIVE, 'processed_archive_sha256': processed_archive_sha256, 'audit_manifest': audit_manifest, 'split_manifest': split_manifest, 'model': {'architecture': 'CNN from scratch: 3 x [Conv-BN-ReLU-Conv-BN-ReLU-MaxPool]', 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'epochs_max': EPOCHS, 'learning_rate': LEARNING_RATE, 'dropout': DROPOUT}, 'tensorflow': tf.__version__}
(OUTPUTS/'metadata'/'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
os.system(f"pip freeze > {OUTPUTS/'metadata'/'environment.txt'}")
print(json.dumps(summary, indent=2))


## Publish

Download `outputs/` về local. Tạo một Hugging Face model repo và upload nguyên thư mục output cùng notebook này; model card phải nêu rõ participant-disjoint protocol, revision dữ liệu, SHA archive và số liệu test.
